In [1]:
import pandas as pd
import sys

sys.path.append('../python_scripts')

run_name = '2d_run_4'



In [ ]:


df_median = pd.read_parquet("../sim_results/{}/median_objectives".format(run_name))
df_median.to_parquet('../sim_results/{}/median_objectives.parquet'.format(run_name))

In [2]:
df = pd.read_parquet('../sim_results/{}/median_objectives.parquet'.format(run_name)).reset_index()

In [3]:
df_train_regret = df.loc[(df['mode'] == 'train') & (df.objective == 'mean_regret')]
df_val_regret = df.loc[(df['mode'] == 'val') & (df.objective == 'mean_regret')]
df_val = df.loc[(df['mode'] == 'val') & (df.objective == 'mean')]
df_train = df.loc[(df['mode'] == 'train') & (df.objective == 'mean')]

s_train = df_train[['gen', 'value']].groupby('gen').agg('mean').iloc[:,0].sort_index()
s_train_regret = df_train_regret[['gen', 'value']].groupby('gen').agg('mean').iloc[:,0].sort_index()
s_val_regret = df_val_regret[['gen', 'value']].groupby('gen').agg('mean').iloc[:, 0].sort_index()
s_val = df_val[['gen', 'value']].groupby('gen').agg('mean').iloc[:, 0].sort_index()


In [4]:
import plotly.graph_objects as go 

fig = go.Figure()
fig.add_trace(go.Scatter(x = s_train_regret.index.values, y = s_train_regret.values, name = 'train_regret'))
fig.add_trace(go.Scatter(x = s_val_regret.index, y = s_val_regret.values, name = 'val_regret'))
fig.add_trace(go.Scatter(x = s_val.index, y = s_val.values, name = 'val'))
fig.add_trace(go.Scatter(x = s_train.index, y = s_train.values, name = 'train'))
fig.show()

In [ ]:
from utils import get_pareto_layers

df_pareto = pd.DataFrame()
df_1  = pd.DataFrame()
for gen in df['gen'].drop_duplicates():
    df_gen = df.loc[df['gen'] == gen]
    df_pivot = pd.pivot_table(df_gen[['mode', 'objective', 'sim_id', 'value']], index = 'sim_id', values = 'value', columns = ['mode', 'objective'])
    df_layers = get_pareto_layers(df_pivot[[('train','mean_regret'), ('train','regret_quantile')]], sense = ['min', 'min'], num_layers=1)
    pareto_indices = df_layers.loc[df_layers.layer==0].index
    df_add = df_pivot.loc[pareto_indices]
    df_add['gen'] = gen
    argmin = int(df_add[('train', 'mean_regret')].argmin())
    df_add_1 = pd.DataFrame(df_add.iloc[argmin]).transpose() 
    df_1 = pd.concat([df_1, df_add_1])
    df_pareto = pd.concat([df_pareto, df_add])
    
df_pareto.to_parquet('../sim_results/{}/train_pareto_by_gen.parquet'.format(run_name))


KeyboardInterrupt: 

In [48]:
#get initial population for new run with max_frac as pymoo variable

from utils import get_pareto_layers, get_dna_hash
import numpy as np

df_gen = df.loc[df['gen'] == max(df['gen'])]
df_pivot = pd.pivot_table(df_gen[['mode', 'objective', 'sim_id', 'value']], index = 'sim_id', values = 'value', columns = ['mode', 'objective'])
df_layers = get_pareto_layers(df_pivot[[('train','mean_regret'), ('train','regret_quantile')]], sense = ['min', 'min'], num_layers=15)
df_layers.layer.value_counts(normalize = True).sort_index().cumsum()
sim_ids = df_layers.loc[df_layers.layer <= 12].index
df1 = pd.read_parquet('s3://jdinvestment/{}/populations/gen_149.parquet'.format(run_name)).loc[sim_ids]
df1['max_frac'] = .05
df2 = pd.read_parquet('s3://jdinvestment/{}/populations/gen_149.parquet'.format(run_name)).loc[sim_ids]
df2['max_frac'] = np.random.uniform(.05, .1, size=df2.shape[0])
df_out = pd.concat([df1, df2]).iloc[:215]
best_index = int(np.where(df_out.index == '2c9631a5d785')[0][0])
df_out.index = df_out.apply(get_dna_hash, axis =1)
df_out.to_parquet('s3://jdinvestment/2d_3obj/populations/gen_0.parquet')



In [49]:
best_index


19

In [16]:
#ALTERNATIVE
from utils import get_pareto_layers

df_pareto = pd.DataFrame()
df_1  = pd.DataFrame()
for gen in sorted(list(df['gen'].drop_duplicates())):
    df_gen = df.loc[df['gen'] == gen]
    df_pivot = pd.pivot_table(df_gen[['mode', 'objective', 'sim_id', 'value']], index = 'sim_id', values = 'value', columns = ['mode', 'objective'])
    #df_layers = get_pareto_layers(df_pivot[[('train','mean_regret'), ('train','regret_quantile')]], sense = ['min', 'min'], num_layers=1)
    #pareto_indices = df_layers.loc[df_layers.layer==0].index
    df_add = df_pivot
    df_add['gen'] = gen
    argmin = int(df_add[('train', 'mean_regret')].argmin())
    df_add_1 = pd.DataFrame(df_add.iloc[argmin]).transpose()
    df_add_1['sim_id'] = df_add_1.index
    df_1 = pd.concat([df_1, df_add_1])
    df_pareto = pd.concat([df_pareto, df_add])
   
    
    
df_pareto.to_parquet('../sim_results/{}/train_pareto_by_gen.parquet'.format(run_name))

In [6]:
import numpy as np
sim_id = df_1.loc[df_1.gen == 149].index[0]
s_values = pd.read_parquet('s3://jdinvestment/{}/portfolio_values/sim_{}.parquet'.format(run_name, sim_id)).transpose().iloc[:-3]
df_pop = pd.read_parquet("s3://jdinvestment/{}/populations/gen_149.parquet".format(run_name))
sim_index = int(np.where(df_pop.index == sim_id)[0][0])
print(sim_index, sim_id, (s_values.iloc[-1]/s_values.iloc[-14])**(1/1), s_values.index[-14], s_values.index[-1])

149 2c9631a5d785 2c9631a5d785    1.186214
dtype: object 2025-05-19_00:00:00 2026-05-18_00:00:00


In [23]:
def find_first_date_after(date_list, target_date):
    # Filter dates strictly greater than the target date, then find the minimum
    future_dates = [d for d in date_list if d > target_date]
    return min(future_dates) if future_dates else None

In [32]:
from datetime import date, timedelta
import pandas as pd
import requests

def get_vgro_prices(val_start_date: date, api_token = '693327461e9541.04731237') -> pd.Series:
    """
    Fetches daily data for VGRO.TO from EODHD starting from val_start_date,
    builds a 28-day stepping schedule, and returns a pandas Series of prices
    indexed by date.
    """
    today = date.today()
    
    # 1. Fetch historical EOD data from EODHD
    # Pulling from val_start_date to today
    url = f"https://eodhistoricaldata.com/api/eod/VGRO.TO"
    params = {
        "api_token": api_token,
        "fmt": "json",
        "from": val_start_date.isoformat(),
        "to": today.isoformat()
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    # Convert to pandas DataFrame
    df = pd.DataFrame(data)
    if df.empty:
        return pd.Series(dtype=float)
        
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    # Use adjusted close ('adjusted_close') if available, otherwise standard 'close'
    price_col = 'adjusted_close' if 'adjusted_close' in df.columns else 'close'
    
    # 2. Generate target 28-day sequence of dates
    target_dates = []
    curr_date = pd.Timestamp(val_start_date)
    end_timestamp = pd.Timestamp(today)
    
    while curr_date <= end_timestamp:
        target_dates.append(curr_date)
        curr_date += timedelta(days=28)
        
    # 3. Match target dates to available trading days (using merge_asof or reindexing)
    # Reindex the dataframe to our target schedule and use 'nearest' or 'ffill' 
    # to find valid trading prices if a 28-day mark falls on a weekend/holiday.
    price_series = df[price_col]
    
    target_index = pd.DatetimeIndex(target_dates)
    
    # Reindex with tolerance to find the closest available trading day (e.g., within 3 days)
    matched_prices = price_series.reindex(target_index, method='nearest', tolerance=pd.Timedelta(days=3))
    
    # Drop any NaNs if a date fell completely outside available bounds
    matched_prices.dropna(inplace=True)
    
    return matched_prices

# Example usage:
# api_token = "YOUR_EODHD_API_KEY"
# start_date = date(2025, 1, 1)
# vgro_series = get_vgro_28day_prices(start_date, api_token)
# print(vgro_series)

In [44]:

df_history = pd.read_parquet('s3://jdinvestment/{}/holdings/sim_{}.parquet'.format(run_name, sim_id))
s_values = df_history.set_index('date').drop('sim_id', axis = 1).sum(1)
df_folds = pd.read_parquet('../strategy/folds_4.parquet')
val_start_date = find_first_date_after(s_values.index, df_folds.loc[df_folds.fold_index == 1].iloc[0].start_date+ pd.Timedelta(days=365)) 
end_date = s_values.index[-1]
num_years = (end_date - val_start_date).days/365.25
s_vgro = get_vgro_prices(val_start_date)

print(num_years, (s_values.loc[end_date]/s_values.loc[val_start_date])**(1/num_years))
print(3, (s_values.iloc[-1]/s_values.iloc[-40])**(1/3))
print(1, (s_values.iloc[-1]/s_values.iloc[-14]))
print(s_values.index[-1])
print((47.04/31.62)**(1/4.5))

3.4496919917864477 1.290441346344342
3 1.321279758067227
1 1.173925822336708
2026-06-15 00:00:00
1.092281414043957


In [42]:
val_start_date + pd.Timedelta(days=365)

Timestamp('2023-01-03 00:00:00')

In [45]:
import plotly.graph_objects as go

fig = go.Figure()
start_date = val_start_date - pd.Timedelta(days = 28)
s_keep = s_values.loc[s_values.index >= val_start_date ]
fig.add_trace(go.Scatter(x = s_keep.index, y = s_keep.values/s_keep.values[0], name = 'algorithm'))
fig.add_trace(go.Scatter(x= s_vgro.index, y = s_vgro.values/s_vgro.values[0], name = 'vgro'))
fig.show()


In [17]:
's3://jdinvestment/{}/holdings/sim_{}.parquet'.format(run_name, sim_id)

's3://jdinvestment/2d_train_to_2025/holdings/sim_8a5b8b45a1c0.parquet'

In [15]:
(s_values.iloc[-1]/s_values.iloc[0])**(13/(len(s_values)-1)), len(s_values)-1

(np.float64(1.2922609296546819), 12)

In [10]:
s_values

,8a5b8b45a1c0
date,
2025-06-16_00:00:00,297444.22
2025-07-14_00:00:00,307416.8367
2025-08-11_00:00:00,314237.0451
2025-09-08_00:00:00,314665.3433
2025-10-06_00:00:00,335636.2684
2025-11-03_00:00:00,342644.3385
2025-12-01_00:00:00,342866.9907
2025-12-29_00:00:00,343344.6647
2026-01-26_00:00:00,359634.9646


In [7]:
s_train_regret = (df_pareto[[('train', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val_regret = (df_pareto[[('val', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val= (df_pareto[[('val', 'mean'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
fig = go.Figure()
fig.add_trace(go.Scatter(x = s_train_regret.index.values, y = s_train_regret.values, name = 'train_regret'))
fig.add_trace(go.Scatter(x = s_val_regret.index, y = s_val_regret.values, name = 'val_regret'))
fig.add_trace(go.Scatter(x = s_val.index, y = s_val.values, name = 'val'))
fig.show()

In [ ]:
df_1.columns

MultiIndex([('train',            'mean'),
            ('train',     'mean_regret'),
            ('train',        'quantile'),
            ('train', 'regret_quantile'),
            (  'val',            'mean'),
            (  'val',     'mean_regret'),
            (  'val',        'quantile'),
            (  'val', 'regret_quantile'),
            (  'gen',                '')],
           names=['mode', 'objective'])

: 

In [8]:
import plotly.graph_objects as go 
s_train_regret = (df_1[[('train', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val_regret = (df_1[[('val', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val = (df_1[[('val', 'mean'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
fig = go.Figure()
fig.add_trace(go.Scatter(x = s_train_regret.index.values, y = s_train_regret.values, name = 'train_regret'))
fig.add_trace(go.Scatter(x = s_val_regret.index, y = s_val_regret.values, name = 'val_regret'))
fig.add_trace(go.Scatter(x = s_val.index, y = s_val.values, name = 'val'))
fig.show()

In [7]:
import xarray as xr
import pandas as pd
s_voo = xr.open_dataset('../simulation_data/momentum.nc').to_array()[0].sel(symbol = 'VOO', band = 'price_end').to_pandas()
fold_index = 4
df_folds = pd.read_parquet('../strategy/folds.parquet')
df_fold = df_folds.loc[df_folds.fold_index == fold_index]

In [5]:
import functools
from objective_functions import mean_annualized_return, WeightedRegimeApplyer, weighted_mean


start_date = min(df_fold.start_date)
end_date = max(df_fold.end_date)
agg_func = functools.partial(mean_annualized_return, start_date, end_date)

voo_func = WeightedRegimeApplyer(df_fold, agg_func, weighted_mean)
voo_mean = voo_func(s_voo)
voo_mean


np.float64(0.15859726102530505)

In [70]:
import functools
from objective_functions import mean_annualized_return, WeightedRegimeApplyer, weighted_mean
df_folds = pd.read_csv('../strategy/folds_1.csv')
for col in ['start_date', 'end_date']:
    df_folds[col] = pd.to_datetime(df_folds[col])
df_voo = pd.read_csv('../simulation_data/voo_backcasted_28days.csv')
s_voo = pd.Series(df_voo.values[:,-1], index = pd.to_datetime(df_voo.iloc[:,0]))
df_out = pd.DataFrame()
for row in range(df_folds.shape[0]):
    df_fold = df_folds.iloc[row].copy()
    start_date, end_date = df_fold[['start_date', 'end_date']]
    

    
    voo_mean = mean_annualized_return(start_date, end_date,s_voo)
    df_fold['voo_return'] = voo_mean
    df_out = pd.concat([df_out, pd.DataFrame(df_fold.copy()).transpose()])
    print(start_date, end_date, voo_mean)
df_out = df_out.rename(columns = {'index': 'description'}).set_index('description')
print(df_out)
        

2008-01-01 00:00:00 2008-08-31 00:00:00 -0.08560702196953351
2008-09-01 00:00:00 2009-03-31 00:00:00 -0.6255475194744984
2009-04-01 00:00:00 2009-12-31 00:00:00 0.4579419345017437
2010-01-01 00:00:00 2011-04-30 00:00:00 0.13197869183563848
2011-05-01 00:00:00 2011-11-30 00:00:00 -0.18230933045622366
2011-12-01 00:00:00 2013-04-30 00:00:00 0.248253698361115
2013-05-01 00:00:00 2013-12-31 00:00:00 0.20293260341316
2014-01-01 00:00:00 2015-07-31 00:00:00 0.10908982197281247
2015-08-01 00:00:00 2016-02-29 00:00:00 -0.022362341330139923
2016-03-01 00:00:00 2017-12-31 00:00:00 0.19914092503598924
2018-01-01 00:00:00 2018-03-31 00:00:00 -0.039952295425778384
2018-04-01 00:00:00 2018-09-30 00:00:00 0.30276171252467443
2018-10-01 00:00:00 2018-12-31 00:00:00 -0.2247335172284438
2019-01-01 00:00:00 2020-01-31 00:00:00 0.2997640483772288
2020-02-01 00:00:00 2020-04-30 00:00:00 -0.3967135888024208
2020-05-01 00:00:00 2021-12-31 00:00:00 0.3420393562675983
2022-01-01 00:00:00 2022-09-30 00:00:00 -0

In [78]:

mid_point = df_out.start_date.min() + pd.Timedelta(days = .75 * (pd.to_datetime('Jan 1, 2025') - df_out.start_date.min()).days)
mid_point

Timestamp('2020-10-01 12:00:00')

In [95]:
df_out.to_csv('../strategy/folds_2.csv')
for col in ['start_date', 'end_date']:
    df_out[col] = pd.to_datetime(df_out[col])

In [97]:
df2 = df_out.copy()

df2['fold_index'] = 1
df2.loc[df2.start_date <= pd.to_datetime('2020-05-01'),'fold_index'] = 0
df2.loc[df2.start_date >= pd.to_datetime('2024-09-01'), 'fold_index'] =2
df2.to_parquet('../strategy/folds_1.parquet')

In [94]:
pd.to_datetime(df_out.end_date)

description
Early GFC Recession (Pre-Lehman)                        2008-08-31
GFC Panic Apex (Lehman to Market Bottom)                2009-03-31
Transition: Liquidity-Driven Early Recovery             2009-12-31
Mid-Cycle Expansion & QE1/QE2                           2011-04-30
Transition: Euro Sovereign Debt & US Downgrade          2011-11-30
Draghi "Whatever It Takes" Stabilization                2013-04-30
Transition: Taper Tantrum Rate Shock                    2013-12-31
US Decoupling & Strong Dollar Inflow                    2015-07-31
Transition: China Devaluation & Oil Crash               2016-02-29
Post-Election Reflation & Low-Vol Goldilocks            2017-12-31
Transition: Volmageddon Short-Volatility Squeeze        2018-03-31
Trade War Escalation & Synchronized Slowdown            2018-09-30
Transition: Fed "Long Way From Neutral" Hawk Crash      2018-12-31
Fed Dovish Pivot & Insurance Rate Cuts                  2020-01-31
Transition: COVID-19 Flash Liquidation Shock      

In [16]:
sim_id = df_1.index[-1]
s3_path = "s3://jdinvestment/{}/portfolio_values/sim_{}.parquet".format(run_name, sim_id)
s_values = pd.read_parquet(s3_path).transpose().iloc[:-2,0]
s_values.index = pd.to_datetime(s_values.index.str.replace('_',' '))
opt_mean = voo_func(s_values)
sim_id, opt_mean, sim_id in df_1.index



('15c81be3b60f', np.float64(-0.03826888266447126), True)

In [155]:
import numpy as np
df_pop = pd.read_parquet('../sim_results/{}/populations/gen_149.parquet'.format(run_name))
sim_index = np.where(df_pop.index == sim_id)[0][0]
sim_index

np.int64(176)

In [142]:
sim_id in df_1.index




True

In [54]:
df_holdings = pd.read_parquet('s3://jdinvestment/2d_test_fold_2/holdings/sim_a99516312234.parquet')

In [102]:
df19 = df_holdings.loc[df_holdings.date.dt.year == 2019].set_index('date')
df19.index = pd.to_datetime(df19.index)
df19.drop('sim_id', axis = 1, inplace = True)
df19 = pd.DataFrame(df19.values/df19.values.sum(1).reshape(df19.shape[0],1), index = df19.index, columns = df19.columns)
df19 = df19.loc[:, df19.max(0) > 0]
df19.mean(0).sort_index(ascending = False)



symbol
XOM     0.007691
XLG     0.003846
WPC     0.003844
WM      0.007689
VRTX    0.003846
VRT     0.008142
TSLA    0.003846
TMUS    0.026552
STAG    0.007692
SCHG    0.042308
RSG     0.008693
REXR    0.011538
PLD     0.015381
PGR     0.015384
ORLY    0.042306
O       0.007691
NNN     0.003846
MSFT    0.011536
MPWR    0.013926
MPC     0.007692
LLY     0.003846
KLAC    0.003846
KKR     0.003848
GILD    0.003844
GIC     0.445235
FR      0.000002
FICO    0.010030
EXEL    0.003846
EGP     0.011533
DECK    0.049999
DBMF    0.003847
CWST    0.046150
CVX     0.011535
CASY    0.000201
BX      0.030769
BTAL    0.011539
BKNG    0.026917
ARES    0.042306
APO     0.019229
APH     0.010338
ADC     0.007692
dtype: float64

In [114]:
df_pop = pd.read_parquet('../sim_results/2d_test_fold_2/populations/gen_149.parquet')
df_pop.shape

(215, 21)

In [115]:
sim_id

'a99516312234'